# Run Evaluation, Test Trajectory Visualization & Conditioned/Unconditioned Generation

Loads a saved joint 2D+3D diffusion run from `./runs/...`, plots training and validation metrics, visualizes reverse-diffusion coordinate trajectories on held-out test molecules (`data/test.parquet`), and demonstrates:

1. **Full 2D SMILES conditioning** (`"CCO"`, `"c1ccccc1"`, `"CC(=O)O"`): conformer generation with full 2D graph conditioning.
2. **Partial template conditioning** (`"c1ccccc1~U~U(=U)"`, `"C1CC1~U~U"`): scaffold-constrained 2D+3D generation where `U` marks unconditioned atoms and `~` marks unconditioned bonds.
3. **Unconditioned generation** (`"U.U.U.U.U"`, `"U.U.U.U.U.U"`): de novo 2D+3D molecule generation from dummy atom templates.

In [ ]:
import json
import pathlib
import tempfile
from typing import Final

from IPython import display as ipython_display
import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
import seaborn as sns

from miniqm import config as config_lib
from miniqm import viz
from miniqm.data import batch as batch_lib
from miniqm.data import data_source
from miniqm.diffusion import geometry, metrics, model, sampling
from miniqm.training import checkpoint, train
from miniqm.training import eval as eval_lib

sns.set_theme(style="whitegrid")

REPO_ROOT: Final = pathlib.Path("..") if pathlib.Path.cwd().name == "notebooks" else pathlib.Path()
DEFAULT_RUN_DIR: Final = REPO_ROOT / "runs" / "qm9_experiment"
TEST_PARQUET: Final = REPO_ROOT / "data" / "test.parquet"
BATCH_SIZE: Final = 32

try:
    run_dir = checkpoint.resolve_run_dir(DEFAULT_RUN_DIR)
    loaded_run = checkpoint.load_run_checkpoint(
        run_dir,
        checkpoint_name=config_lib.CheckpointName.BEST,
    )
except FileNotFoundError:
    demo_parent = pathlib.Path(tempfile.mkdtemp(prefix="miniqm_demo_run_"))
    print(f"No saved run at {DEFAULT_RUN_DIR}; running a 20-step demo job in {demo_parent}...")
    train.run_training(
        config_lib.RunConfig(
            train_path=REPO_ROOT / "data" / "train.parquet",
            val_path=REPO_ROOT / "data" / "validation.parquet",
            run_dir=demo_parent,
            batch_size=16,
            num_steps=20,
            log_every=5,
            eval_every=10,
            sample_every=10,
            max_train_molecules=64,
            max_val_molecules=32,
            sample_batch_size=8,
            train=config_lib.TrainConfig(
                model=config_lib.DenoiserConfig(
                    transformer=config_lib.TransformerConfig(
                        hidden_dim=64,
                        num_layers=2,
                        num_heads=4,
                    ),
                ),
                learning_rate=1e-3,
                warmup_steps=5,
                decay_steps=20,
                ema_decay=0.9,
            ),
            sampling=config_lib.SamplingConfig(num_steps=20),
        )
    )
    run_dir = checkpoint.resolve_run_dir(demo_parent)
    loaded_run = checkpoint.load_run_checkpoint(
        run_dir,
        checkpoint_name=config_lib.CheckpointName.BEST,
    )

print(
    f"Loaded checkpoint from {loaded_run.run_dir} at step={loaded_run.checkpoint.step} "
    f"({train.count_parameters(loaded_run.checkpoint.ema_params):,} parameters)."
)

In [ ]:
metrics_path = run_dir / "metrics.jsonl"
records = tuple(
    map(
        json.loads,
        filter(None, map(str.strip, metrics_path.read_text(encoding="utf-8").splitlines())),
    )
)
metrics_df = pd.DataFrame(records)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

loss_df = metrics_df[metrics_df["split"].isin(("train", "val"))]
sns.lineplot(
    data=loss_df,
    x="step",
    y="total_loss",
    hue="split",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Total Diffusion Loss vs. Step")
axes[0].set_xlabel("Optimization Step")
axes[0].set_ylabel("Total Loss")

sns.lineplot(
    data=loss_df,
    x="step",
    y="rmsd",
    hue="split",
    marker="o",
    ax=axes[1],
)
sample_df = metrics_df[metrics_df["split"] == "val_sample"]
if not sample_df.empty:
    axes[1].plot(
        sample_df["step"],
        sample_df["sampled_rmsd_mean"],
        marker="s",
        linestyle="--",
        color="tab:green",
        label="val_sample (reverse ODE)",
    )
    axes[1].legend()
axes[1].set_title("Coordinate RMSD (Å) vs. Step")
axes[1].set_xlabel("Optimization Step")
axes[1].set_ylabel("Kabsch RMSD (Å)")

if "atom_loss" in loss_df.columns and "bond_loss" in loss_df.columns:
    val_df = loss_df[loss_df["split"] == "val"]
    axes[2].plot(val_df["step"], val_df["atom_loss"], marker="o", label="Val Atom Loss")
    axes[2].plot(val_df["step"], val_df["bond_loss"], marker="s", label="Val Bond Loss")
    axes[2].set_title("2D Atom & Bond Loss vs. Step")
    axes[2].set_xlabel("Optimization Step")
    axes[2].set_ylabel("EDM-Weighted MSE")
    axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
test_source = data_source.DataSource(TEST_PARQUET, max_molecules=BATCH_SIZE)
test_loss = eval_lib.evaluate_loss(
    loaded_run.checkpoint.ema_params,
    jax.random.key(0),
    test_source,
    batch_size=8,
    model_config=loaded_run.model_config,
    loss_config=loaded_run.loss_config,
)
test_sampling = eval_lib.evaluate_dataset_sampling(
    loaded_run.checkpoint.ema_params,
    jax.random.key(1),
    test_source,
    batch_size=8,
    model_config=loaded_run.model_config,
    sampling_config=loaded_run.sampling_config,
)
print(
    f"Test split metrics ({len(test_source)} molecules): "
    f"loss={test_loss.total_loss:.4f}, teacher_forced_rmsd={test_loss.rmsd:.3f} Å, "
    f"atom_loss={test_loss.atom_loss:.4f}, bond_loss={test_loss.bond_loss:.4f}, "
    f"sampled_rmsd_mean={test_sampling.sampled_rmsd_mean:.3f} Å, "
    f"sampled_lddt_mean={test_sampling.sampled_lddt_mean:.3f}, "
    f"stereo_violations={100.0 * test_sampling.stereocenter_violation_rate:.1f}%"
)

host_test_batch = next(test_source.iter_batches(batch_size=BATCH_SIZE, drop_last=False))
jax_test_batch = host_test_batch.to_jax()
denoiser = model.Denoiser(config=loaded_run.model_config)
test_trajectory = sampling.sample_trajectory(
    jax.random.key(42),
    denoiser,
    loaded_run.checkpoint.ema_params,
    batch=jax_test_batch,
    config=loaded_run.sampling_config,
)
generated_test_positions = test_trajectory[-1]
aligned_test_positions = geometry.kabsch_align(
    reference=jax_test_batch.positions,
    target=generated_test_positions,
    atom_mask=jax_test_batch.atom_mask,
)
per_mol_rmsd = metrics.rmsd_per_molecule(
    predicted_positions=generated_test_positions,
    target_positions=jax_test_batch.positions,
    atom_mask=jax_test_batch.atom_mask,
)
per_mol_lddt = metrics.lddt_per_molecule(
    predicted_positions=generated_test_positions,
    target_positions=jax_test_batch.positions,
    atom_mask=jax_test_batch.atom_mask,
)

In [ ]:
selected_idx = np.random.default_rng().integers(0, BATCH_SIZE)
mol_rmsd = float(per_mol_rmsd[selected_idx])
mol_lddt = float(per_mol_lddt[selected_idx])
print(
    f"Test molecule {selected_idx}: Kabsch RMSD = {mol_rmsd:.3f} Å, lDDT = {mol_lddt:.3f} "
    "(translucent = ground truth, cyan = generated conformer)"
)
overlay_viewer = viz.show_overlay(
    reference_positions=np.asarray(jax_test_batch.positions[selected_idx]),
    candidate_positions=np.asarray(aligned_test_positions[selected_idx]),
    atom_features=host_test_batch.atom_features[selected_idx],
    atom_mask=host_test_batch.atom_mask[selected_idx],
)
overlay_viewer.show()

In [ ]:
num_test_frames, *_ = test_trajectory.shape
print(
    f"Animated reverse-diffusion trajectory ({num_test_frames} frames) "
    f"for test molecule {selected_idx}"
)
trajectory_viewer = viz.show_trajectory(
    trajectory=np.asarray(test_trajectory[:, selected_idx]),
    atom_features=host_test_batch.atom_features[selected_idx],
    atom_mask=host_test_batch.atom_mask[selected_idx],
    interval_ms=80,
)
trajectory_viewer.show()

In [ ]:
TEMPLATES: Final = (
    # Full 2D SMILES conditioning (model predicts 3D conformers):
    "CCO",
    "c1ccccc1",
    "CC(=O)O",
    # Partial template conditioning (U = dummy atom, ~ = dummy bond):
    "c1ccccc1~U~U(=U)",
    "C1CC1~U~U",
    # Unconditioned de novo 2D+3D generation (all atoms and bonds unconditioned):
    "U.U.U.U.U",
    "U.U.U.U.U.U",
)

template_batch = batch_lib.collate_smiles(TEMPLATES)
template_mol_trajectory = sampling.sample_molecule_trajectory(
    jax.random.key(7),
    denoiser,
    loaded_run.checkpoint.ema_params,
    batch=template_batch.to_jax(),
    config=loaded_run.sampling_config,
)
sampled_positions = np.asarray(template_mol_trajectory.positions[-1])
sampled_atoms = np.asarray(template_mol_trajectory.atom_features[-1])
sampled_bonds = np.asarray(template_mol_trajectory.bond_features[-1])

generated_mols = tuple(
    batch_lib.build_generated_mol(
        positions=sampled_positions[idx],
        atom_features=sampled_atoms[idx],
        bond_features=sampled_bonds[idx],
        atom_mask=template_batch.atom_mask[idx],
        cond_atom_features=template_batch.atom_features[idx],
        atom_cond_mask=template_batch.atom_cond_mask[idx],
        cond_bond_features=template_batch.bond_features[idx],
        bond_cond_mask=template_batch.bond_cond_mask[idx],
        prompt_smiles=TEMPLATES[idx],
    )
    for idx in range(len(TEMPLATES))
)
legends = tuple(
    f"prompt: {prompt}\ngen: {Chem.MolToSmiles(mol)}"
    for prompt, mol in zip(TEMPLATES, generated_mols, strict=True)
)
for prompt, mol in zip(TEMPLATES, generated_mols, strict=True):
    print(f"Template: {prompt:<20s} -> Generated SMILES: {Chem.MolToSmiles(mol)}")

ipython_display.display(
    Draw.MolsToGridImage(
        list(generated_mols),
        molsPerRow=4,
        subImgSize=(260, 220),
        legends=list(legends),
    )
)

demo_idx = 3  # "c1ccccc1~U~U(=U)" partial template
print(
    f"\n3D trajectory for template {TEMPLATES[demo_idx]!r} -> "
    f"generated SMILES {Chem.MolToSmiles(generated_mols[demo_idx])!r}"
)
template_viewer = viz.show_trajectory(
    trajectory=np.asarray(template_mol_trajectory.positions[:, demo_idx]),
    atom_features=np.asarray(template_mol_trajectory.atom_features[:, demo_idx]),
    atom_mask=template_batch.atom_mask[demo_idx],
    cond_atom_features=template_batch.atom_features[demo_idx],
    atom_cond_mask=template_batch.atom_cond_mask[demo_idx],
    cond_bond_features=template_batch.bond_features[demo_idx],
    bond_cond_mask=template_batch.bond_cond_mask[demo_idx],
    interval_ms=80,
)
template_viewer.show()